In [1]:
#Assignment-1
#Name: Ashlesha Tayade            Name: Ruchika Gaikwad
#PRN: 260240128008                PRN: 260240128036

In [2]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [3]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [4]:
#2. Create a Pair RDD from a text file using map()
#a
input_file = f"/home/talentum/test-jupyter/P2/M2/SM4/Pair-RDD/selfishgiant.txt"

splitRdd = sc.textFile(f"FILE://{input_file}").flatMap(lambda line: line.split(" "))
splitRdd.take(5)


['EVERY', 'afternoon,', 'as', 'they', 'were']

In [5]:
#b
mappedRdd = splitRdd.map(lambda word: (word, 1))
mappedRdd.take(5)

[('EVERY', 1), ('afternoon,', 1), ('as', 1), ('they', 1), ('were', 1)]

In [6]:
#3.Create Pair RDDs using zip functions and perform simple transformations
#a

months = ("Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul")

monthsRdd = sc.parallelize(months)
monthsIndexed0Rdd = monthsRdd.zipWithIndex()

print(monthsIndexed0Rdd.collect())

[('Jan', 0), ('Feb', 1), ('Mar', 2), ('Apr', 3), ('May', 4), ('Jun', 5), ('Jul', 6)]


In [7]:
#b

monthsIndexed1Rdd = monthsIndexed0Rdd.map(lambda item: (item[0], item[1]+1))
print(monthsIndexed1Rdd.collect())

[('Jan', 1), ('Feb', 2), ('Mar', 3), ('Apr', 4), ('May', 5), ('Jun', 6), ('Jul', 7)]


In [8]:
#c
monthsIndexed2Rdd = monthsIndexed0Rdd.mapValues(lambda y: y+1)
print(monthsIndexed2Rdd.collect())

[('Jan', 1), ('Feb', 2), ('Mar', 3), ('Apr', 4), ('May', 5), ('Jun', 6), ('Jul', 7)]


In [9]:
#d
quarters = (1, 1, 1, 2, 2, 2, 3)

quartersRdd = sc.parallelize(quarters)

monthsZipQuarters = monthsRdd.zip(quartersRdd)
print(monthsZipQuarters.collect())

[('Jan', 1), ('Feb', 1), ('Mar', 1), ('Apr', 2), ('May', 2), ('Jun', 2), ('Jul', 3)]


In [10]:
#e
print(monthsZipQuarters.keys().collect())
print(monthsZipQuarters.values().collect())
print(monthsZipQuarters.sortByKey().collect())

['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul']
[1, 1, 1, 2, 2, 2, 3]
[('Apr', 2), ('Feb', 1), ('Jan', 1), ('Jul', 3), ('Jun', 2), ('Mar', 1), ('May', 2)]


In [11]:
#4. Count the number of times words appear in a Pair RDD and manipulate the output
#a
reducedByKeyRdd = mappedRdd.reduceByKey(lambda x, y: x + y)
print(reducedByKeyRdd.take(5))

[('EVERY', 1), ('as', 9), ('school,', 1), ('used', 4), ('go', 1)]


In [12]:
#b
flippedRdd = reducedByKeyRdd.map(lambda item: (item[1], item[0]))
flippedRdd.take(5)

[(1, 'EVERY'), (9, 'as'), (1, 'school,'), (4, 'used'), (1, 'go')]

In [13]:
#c
orderedRdd = flippedRdd.sortByKey(ascending=False)
orderedRdd.take(5)

[(148, 'the'), (85, 'and'), (44, 'he'), (38, 'to'), (33, '')]

In [14]:
#Challenge Labs
#d
input_file = f"/home/talentum/test-jupyter/P2/M2/SM4/Pair-RDD/flights.csv"

carrierRdd = sc.textFile(f"FILE://{input_file}").map(lambda line: line.split(",")).map(lambda cols: (cols[5], 1))
carrierRdd.take(5)


[('WN', 1), ('WN', 1), ('WN', 1), ('WN', 1), ('WN', 1)]

In [15]:
#e

carrierSorted = carrierRdd.reduceByKey(lambda x, y: x+y).map(lambda items: (items[1], items[0])).sortByKey(ascending=False)
print(carrierSorted.take(5))

[(356167, 'WN'), (175969, 'AA'), (166445, 'OO'), (141178, 'MQ'), (133403, 'US')]


In [16]:
#2. Determine the most common routes between two cities
#c

input_file = f"/home/talentum/test-jupyter/P2/M2/SM4/Pair-RDD/airports.csv"

cityRdd = sc.textFile(f"FILE://{input_file}").map(lambda line: line.split(",")).map(lambda cols: (cols[0], cols[2]))
print(cityRdd.take(5))

[('iata', 'city'), ('00M', 'BaySprings'), ('00R', 'Livingston'), ('00V', 'ColoradoSprings'), ('01G', 'Perry')]


In [17]:
#d

input_file = f"/home/talentum/test-jupyter/P2/M2/SM4/Pair-RDD/flights.csv"

flightOrigDestRdd = sc.textFile(f"FILE://{input_file}").map(lambda line: line.split(",")).map(lambda cols: (cols[12], cols[13]))
flightOrigDestRdd.take(5)

[('IAD', 'TPA'),
 ('IND', 'BWI'),
 ('IND', 'JAX'),
 ('IND', 'LAS'),
 ('IND', 'PHX')]

In [18]:
#e

origJoinRdd = flightOrigDestRdd.join(cityRdd)
print(origJoinRdd.take(5))

[('ONT', ('LAS', 'Ontario')), ('ONT', ('LAS', 'Ontario')), ('ONT', ('OAK', 'Ontario')), ('ONT', ('OAK', 'Ontario')), ('ONT', ('OAK', 'Ontario'))]


In [19]:
#f

destOrigJoinRdd = origJoinRdd.values().join(cityRdd)
print(destOrigJoinRdd.take(5))

[('LAS', ('Ontario', 'LasVegas')), ('LAS', ('Ontario', 'LasVegas')), ('LAS', ('Ontario', 'LasVegas')), ('LAS', ('Ontario', 'LasVegas')), ('LAS', ('Ontario', 'LasVegas'))]


In [20]:
#e

cityCleanedRdd = destOrigJoinRdd.values()
print(cityCleanedRdd.take(5))

[('Ontario', 'LasVegas'), ('Ontario', 'LasVegas'), ('Ontario', 'LasVegas'), ('Ontario', 'LasVegas'), ('Ontario', 'LasVegas')]


In [21]:
#h

citiesKV = cityCleanedRdd.map(lambda cities: (cities, 1))
print(citiesKV.take(5))

[(('Ontario', 'LasVegas'), 1), (('Ontario', 'LasVegas'), 1), (('Ontario', 'LasVegas'), 1), (('Ontario', 'LasVegas'), 1), (('Ontario', 'LasVegas'), 1)]


In [22]:
#i

citiesReducedSortedRdd = citiesKV.reduceByKey(lambda x, y: x + y).map(lambda items: (items[1], items[0])).sortByKey(ascending=False)
print(citiesReducedSortedRdd.take(3))

[(5540, ('NewYork', 'Boston')), (5478, ('Boston', 'NewYork')), (4103, ('Chicago', 'NewYork'))]


In [23]:
#3.Find the longest departure delays for any airline that experienced a delay of 15 minutes or more
#b
input_file = f"/home/talentum/test-jupyter/P2/M2/SM4/Pair-RDD/flights.csv"

delayRdd = sc.textFile(f"FILE://{input_file}").map(lambda line: line.split(",")).filter(lambda delay: int(delay[11]) > 15).map(lambda cols: (cols[5], cols[11]))


delayRdd.take(5)

[('WN', '25'), ('WN', '67'), ('WN', '87'), ('WN', '29'), ('WN', '82')]

In [24]:
#c

deplayMaxRdd = delayRdd.reduceByKey(lambda x, y: max(int(x), int(y)))
print(deplayMaxRdd.take(5))

[('XE', 781), ('YV', 526), ('OH', 680), ('OO', 767), ('UA', 1268)]


In [25]:
#4.Remove records than contain incomplete data from a file
#c

input_file = f"/home/talentum/test-jupyter/P2/M2/SM4/Pair-RDD/plane-data.csv"

planeDataRdd = sc.textFile(f"FILE://{input_file}")

print(planeDataRdd.count())

5030


In [26]:
#d

cleanedPlaneDataRdd = planeDataRdd.map(lambda val: val.split(",")).filter(lambda ele: len(ele) == 9)
print(cleanedPlaneDataRdd.count())

4481
